# 15 · Scaling sweeps in D, T and d_max  (~6–10 h total; each sweep runs on its own)

> Part of the October 2026 experiments (notebooks 10–19), run with the fastapriori 0.2.0 build and the `bench/exp_runners.py` harness. Defaults: 1 warm-up + 3 measured runs per cell, full repetitions even for slow cells, explicit thread counts, failures recorded. Never run two of these notebooks at the same time.

Feeds §6.4 (new figure). Quest-model data from `exp_data.quest` (vectorised version of
`bench/agrawal_generator.py`: same model, different random stream), s = 1e-4, k ∈ {2,3,4}.

* **D sweep**: T=20, I=4, N=50K, L=50K, corr 0.5; D ∈ {0.1, 0.25, 0.5, 1, 2.5, 5} M.
  P2 predicts the 1-thread sparse layout loses ground to the triangular matrix as D grows (ρ ∝ 1/D).
* **T sweep**: D=500K, N=25K, I=4, L=25K; T ∈ {5, 10, 20, 30, 50}.
* **d_max sweep**: base T=10, D=500K, N=25K; `inject_tail` raises d_max ∈ {50, 200, 500,
  1000, 2500} at fixed D and d_avg (d_CV rises with it, as it must). Tests the C(d_max, k−1) term.

Set `RUN_SWEEPS`. Generation is cached in `data/generated/`; delete a parquet to regenerate.

Outputs: `results/runs/<campaign>/scaling_D.csv`, `scaling_T.csv`, `scaling_dmax.csv`.

In [1]:
import os, sys, json, math, time
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
sys.path.insert(0, str(BUNDLE / "bench"))
# Results of these notebooks go to results/runs/<FA_CAMPAIGN>/ (set it before starting Jupyter
# to keep campaigns apart; every run_grid call also appends to manifest.jsonl there).
os.environ.setdefault("FA_CAMPAIGN", "new_campaign")
import numpy as np, pandas as pd
import exp_runners as pr
import exp_data as pdd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
print("build", pr.build_id(), "| extension", pr.ext_path())

campaign vm_20261029 -> /home/user/artifacts/results/runs/vm_20261029
build cc508b6408f4 | extension /home/user/.venvs/fastapriori/lib/python3.12/site-packages/fastapriori/_fastapriori_rs.cpython-312-x86_64-linux-gnu.so


In [2]:
RUN_SWEEPS = ["T"]     # d_max first: it is the decisive one
METHODS = ["hyb", "hyb_1t", "hyb_sfx_1t", "hyb_seeded", "hyb_seeded_1t",
           "pyfim_eclat", "pyfim_fpgrowth", "borgelt_apriori", "borgelt_eclat", "lcm"]
KS, S = [2, 3, 4], 1e-4

def make(name, gen):
    if not pdd.is_registered(name):
        t0 = time.time(); df = gen()
        st = pdd.cost_stats(df, (S,))[0]
        pdd.register_parquet(name, df, meta=st)
        print(f"generated {name}: {len(df):,} rows in {time.time()-t0:.0f}s, d_avg={st['d_avg']:.2f}, d_max={st['d_max']}")
    return name

In [3]:
if "dmax" in RUN_SWEEPS:
    base = pdd.quest(D=500_000, T=10, I=4, L=25_000, N=25_000, seed=11)
    names = [make(f"sweep_dmax_{dm}", lambda dm=dm: pdd.inject_tail(base, d_max=dm, n_wide=100))
             for dm in (50, 200, 500, 1000, 2500)]
    pr.run_grid("scaling_dmax", names, METHODS, ks=KS, supports=[S], timeout_s=1500)

In [4]:
if "D" in RUN_SWEEPS:
    names = [make(f"sweep_D_{D}", lambda D=D: pdd.quest(D=D, T=20, I=4, L=50_000, N=50_000, seed=12))
             for D in (100_000, 250_000, 500_000, 1_000_000, 2_500_000, 5_000_000)]
    pr.run_grid("scaling_D", names, METHODS, ks=KS, supports=[S], timeout_s=1500)

In [5]:
if "T" in RUN_SWEEPS:
    names = [make(f"sweep_T_{T}", lambda T=T: pdd.quest(D=500_000, T=T, I=4, L=25_000, N=25_000, seed=13))
             for T in (5, 10, 20, 30, 50)]
    pr.run_grid("scaling_T", names, METHODS, ks=KS, supports=[S], timeout_s=1500)

generated sweep_T_5: 2,502,721 rows in 2s, d_avg=5.01, d_max=18


generated sweep_T_10: 5,001,348 rows in 4s, d_avg=10.00, d_max=27


generated sweep_T_20: 10,002,374 rows in 7s, d_avg=20.00, d_max=43


generated sweep_T_30: 14,996,457 rows in 11s, d_avg=29.99, d_max=57


generated sweep_T_50: 24,979,019 rows in 17s, d_avg=49.96, d_max=85
preflight ok: 10 methods, build cc508b6408f4


scaling_T:   0%|          | 0/150 [00:00<?, ?cell/s]

... (596 outputs left out)

[scaling_T] sweep_T_50 | lcm | k=4 s=0.0001 run 0: ok wall=16.57 rss=235668.0 (16.6s)


[scaling_T] sweep_T_50 | lcm | k=4 s=0.0001 run 1: ok wall=16.34 rss=235612.0 (16.3s)


[scaling_T] sweep_T_50 | lcm | k=4 s=0.0001 run 2: ok wall=16.22 rss=235604.0 (16.2s)


[scaling_T] sweep_T_50 | lcm | k=4 s=0.0001 run 3: ok wall=16.29 rss=235676.0 (16.3s)


In [6]:
for exp in ("scaling_dmax", "scaling_D", "scaling_T"):
    s = pr.summarize(pr.load_results(exp))
    if len(s):
        display(exp, s.pivot_table(index=["dataset", "k"], columns="method", values="wall_s").round(2))

'scaling_dmax'

method             borgelt_apriori  borgelt_eclat   hyb  hyb_1t  hyb_seeded  hyb_seeded_1t  hyb_sfx_1t   lcm  pyfim_eclat  pyfim_fpgrowth
dataset         k                                                                                                                        
sweep_dmax_1000 2             2.28           1.44  0.52    1.18        0.52           1.19        0.97  1.78         1.45            1.71
                3             2.96           1.52  1.05    2.31        0.51           1.09        2.30  1.77         1.53            1.72
                4             3.02           1.52  1.24    2.50        0.51           1.10        2.48  1.77         1.52            1.70
sweep_dmax_200  2             2.14           1.39  0.49    1.00        0.49           1.01        0.86  1.57         1.41            1.63
                3             2.82           1.48  1.03    2.11        0.49           0.99        2.10  1.55         1.47            1.58
                4             2.92           1.47  1.20    2.32        0.49           1.01        2.28  1.57         1.49            1.56
sweep_dmax_2500 2             2.48           1.62  0.70    2.23        0.70           2.21        1.57  2.56         1.64            2.01
                3             3.20           1.73  1.33    3.84        0.59           1.53        3.85  2.58         1.72            2.01
                4             3.28           1.71  1.49    4.05        0.60           1.54        4.02  2.58         1.72            2.10
sweep_dmax_50   2             2.15           1.41  0.48    1.00        0.48           1.00        0.85  1.59         1.41            1.56
                3             2.86           1.48  1.01    2.10        0.49           0.98        2.08  1.56         1.48            1.56
                4             2.91           1.48  1.20    2.27        0.49           1.00        2.25  1.55         1.46            1.54
sweep_dmax_500  2             2.20           1.41  0.49    1.04        0.49           1.04        0.89  1.62         1.43            1.64
                3             2.89           1.48  1.02    2.11        0.50           1.02        2.13  1.67         1.49            1.61
                4             2.97           1.50  1.20    2.29        0.50           1.03        2.32  1.64         1.48            1.63

'scaling_D'

method             borgelt_apriori  borgelt_eclat    hyb  hyb_1t  hyb_seeded  hyb_seeded_1t  hyb_sfx_1t    lcm  pyfim_eclat  pyfim_fpgrowth
dataset         k                                                                                                                          
sweep_D_100000  2             6.80           2.16   0.23    0.48        0.24           0.48        0.41   0.65         2.25            1.00
                3             9.12           2.51   0.47    0.97        0.24           0.48        0.97   0.66         2.62            1.01
                4             9.19           2.53   0.53    1.07        0.23           0.48        1.07   0.66         2.64            1.01
sweep_D_1000000 2            11.30           8.10   2.48    8.81        2.46           7.76        7.43  10.49         8.36           10.25
                3            15.47           8.63   4.87   15.66        2.37           6.57       15.80  10.26         8.45           10.45
                4            15.50           8.62   5.73   17.38        2.31           7.90       15.19  10.54         8.87           10.48
sweep_D_250000  2             7.80           2.93   0.53    1.23        0.53           1.23        1.04   2.31         2.74            2.79
                3            10.39           3.26   1.03    2.46        0.54           1.19        2.43   2.29         3.08            2.78
                4            10.48           3.27   1.23    2.71        0.53           1.20        2.70   2.34         3.26            2.54
sweep_D_2500000 2            18.78          18.20   6.45   22.25        6.26          21.59       17.72  28.16        17.65           28.24
                3            24.50          19.23  12.59   39.07        6.16          18.84       37.26  28.24        18.79           27.92
                4            26.01          19.27  14.98   42.60        6.11          21.15       40.57  28.57        18.65           27.83
sweep_D_500000  2             9.20           5.01   1.15    3.48        1.13           3.02        2.57   5.16         4.94            5.01
                3            12.04           5.43   2.27    5.69        1.13           2.82        5.65   5.18         5.42            5.08
                4            12.45           5.47   2.62    6.74        1.09           2.70        6.24   5.17         5.39            5.04
sweep_D_5000000 2            30.41          36.25  13.21   45.36       13.12          45.45       39.13  58.15        35.83           63.33
                3            40.32          38.17  25.25   81.78       12.83          41.90       81.78  57.93        37.09           63.10
                4            42.13          38.15  30.20   89.14       12.85          42.42       88.90  58.31        36.98           63.50

'scaling_T'

method        borgelt_apriori  borgelt_eclat   hyb  hyb_1t  hyb_seeded  hyb_seeded_1t  hyb_sfx_1t    lcm  pyfim_eclat  pyfim_fpgrowth
dataset    k                                                                                                                         
sweep_T_10 2             2.18           1.51  0.49    1.02        0.49           1.01        0.86   1.71         1.49            1.73
           3             2.89           1.51  1.06    2.18        0.51           1.04        2.22   1.66         1.59            1.71
           4             2.94           1.52  1.25    2.34        0.51           1.03        2.40   1.69         1.58            1.70
sweep_T_20 2             3.69           3.59  1.04    3.01        1.04           2.87        2.32   5.10         3.28            4.57
           3             5.91           4.14  2.35    7.03        1.06           2.90        6.88   5.11         3.85            4.87
           4             6.52           4.18  2.87    8.00        1.06           2.88        7.99   5.09         3.85            4.86
sweep_T_30 2             5.51           5.47  1.72    5.97        1.71           5.28        4.49   8.08         5.35            9.39
           3             9.06           6.37  4.01   14.34        1.71           5.23       13.67   8.14         6.25            9.30
           4            10.37           6.52  4.96   16.18        1.70           5.47       16.40   8.15         6.30            9.25
sweep_T_5  2             1.28           0.57  0.26    0.42        0.25           0.43        0.36   0.56         0.61            0.67
           3             1.44           0.55  0.43    0.64        0.26           0.38        0.63   0.57         0.58            0.66
           4             1.44           0.50  0.47    0.67        0.25           0.37        0.68   0.56         0.54            0.61
sweep_T_50 2            11.01          10.47  2.95   10.36        2.96          10.26        9.66  16.57        10.38           24.35
           3            18.48          12.25  8.00   31.31        3.44          13.35       31.82  16.62        11.77           24.99
           4            21.38          12.59  9.91   38.38        3.59          14.32       40.64  16.29        12.44           24.57